<a href="https://colab.research.google.com/github/SlavaMarina/ib-cs-ml-course/blob/main/week4_neural_networks/Week4_Lesson8_Workshop.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 IB CS — Week 4 · Lesson 8 (Workshop)
## Lab "Brain in Code" — neural network from scratch + MNIST CNN
### A4.3.8 + A4.3.9 in practice

> ⏱ Duration: ~90 minutes (a double lesson).
> 💻 Platform: Google Colab; **GPU** is recommended for CNN, but not required.
> 🎯 Goal: understand **how a neural network works** by implementing it **by hand** in NumPy, then build and train a **real CNN** on MNIST.

---

### 📋 Workshop plan

| Part | Topic | Time |
|---|---|---|
| 1 | Forward pass by hand in NumPy (one neuron) | 15 min |
| 2 | Simple ANN in NumPy (3 layers, OR/AND logic gates) | 20 min |
| 3 | Introduction to MNIST (visualisation) | 15 min |
| 4 | Train an **ANN** on MNIST with Keras | 25 min |
| 5 | Train a **CNN** on MNIST | 25 min |
| 6 | Compare ANN vs CNN + error analysis | 15 min |
| 7 | IB-style conclusion | 5 min |

---

### ⚠️ Preparation (run in the first Colab cell)

```python
# If TensorFlow is not installed:
# In Colab, it is installed by default.
# Locally: pip install tensorflow-cpu
```


In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Disable noisy warnings
import warnings
warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

np.random.seed(42)
tf.random.set_seed(42)

print(f"✅ TensorFlow version: {tf.__version__}")
print(f"✅ Keras version: {keras.__version__}")
print(f"💻 GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")


## Part 1 · Forward Pass of One Neuron in NumPy

> We reproduce the **worked example from Baumgarten (p. 47)** by hand.


In [ ]:
# One neuron. Inputs, weights, and bias are given.
inputs  = np.array([1.3, 4.2, 0.0, 2.7])
weights = np.array([-3.1, 1.6, 2.9, 2.7])
bias    = -5.2

# Steps 1-3: weighted sum, which is matrix multiplication
summation = np.dot(inputs, weights)
print(f"Inputs:  {inputs}")
print(f"Weights: {weights}")
print(f"Bias:    {bias}")
print(f"\nSteps 1-3 — Weighted sum:")
print(f"  np.dot(inputs, weights) = {summation:.4f}")

# Step 4: add bias
z = summation + bias
print(f"\nStep 4 — Add bias: {summation:.4f} + ({bias}) = {z:.4f}")

# Step 5: ReLU activation
def relu(x):
    return np.maximum(0, x)

output = relu(z)
print(f"\nStep 5 — ReLU({z:.4f}) = {output:.4f}")
print(f"\n💎 Neuron output: {output:.4f}")


### 🎯 TRY IT #1 — Experiment

Change the weights and bias. How does the output change?

- If bias = +5.0 instead of -5.2, what is the output?
- If all weights are negative, what is the output?

> 💡 ReLU turns negative values into zero. This is critical to understand.


## Part 2 · Simple ANN in NumPy — Logic Gates (OR / AND)

> From Baumgarten (p. 49): a network with 2 inputs → 4 hidden neurons → 2 outputs for logic gates.

**Architecture:**
- 2 inputs: A, B
- 4 neurons in the hidden layer, with ReLU
- 2 outputs: OR, AND, with Sigmoid


In [ ]:
# Weights and biases are already trained (Baumgarten, p. 50)
# Hidden layer
W1 = np.array([
    [ 0.60, -0.47,  0.70,  2.10],
    [-1.13, -1.11,  2.10,  0.80],
])
b1 = np.array([0.53, 0.00, 0.00, -0.23])

# Output layer
W2 = np.array([
    [1.20, -0.30],
    [-0.20, -0.50],
    [1.50, 0.10],
    [0.40, 0.90],
])
b2 = np.array([-1.0, -1.5])  # chosen so both gates work

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def forward_pass(X, verbose=False):
    # Forward pass for inputs X, a vector of 2 values
    # Layer 1
    z1 = X @ W1 + b1
    a1 = relu(z1)
    if verbose:
        print(f"  Hidden activations:  {a1.round(2)}")

    # Layer 2
    z2 = a1 @ W2 + b2
    a2 = sigmoid(z2)
    if verbose:
        print(f"  Output activations:  {a2.round(2)}")
        print(f"  Rounded prediction:  OR={round(a2[0])}, AND={round(a2[1])}")
    return a2

# Test all 4 input combinations
print("=" * 60)
print(f"{'Input':<12} | {'Expected':<14} | {'Predicted':<14}")
print("-" * 60)
for A, B in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    X = np.array([A, B])
    out = forward_pass(X)
    expected_or = A | B
    expected_and = A & B
    pred_or, pred_and = round(out[0]), round(out[1])
    print(f"A={A}, B={B}     | OR={expected_or}, AND={expected_and}    | OR={pred_or}, AND={pred_and}")
print("=" * 60)
print("\n💎 If all rows match, the network works.")


In [ ]:
# Detailed walkthrough of one forward pass: A=1, B=0
print("=== Detailed forward pass for input [1, 0] ===\n")

X = np.array([1.0, 0.0])
print(f"Input: {X}\n")

print("Hidden layer, 4 neurons:")
for i in range(4):
    z_i = X[0]*W1[0,i] + X[1]*W1[1,i] + b1[i]
    a_i = relu(z_i)
    print(f"  H{i+1}: ReLU(1·{W1[0,i]:+.2f} + 0·{W1[1,i]:+.2f} + {b1[i]:+.2f}) = ReLU({z_i:+.2f}) = {a_i:.2f}")

print("\nOutput layer, 2 neurons:")
a1 = relu(X @ W1 + b1)
for i in range(2):
    z_i = (a1 * W2[:, i]).sum() + b2[i]
    a_i = sigmoid(z_i)
    print(f"  O{i+1}: sigmoid(...) = sigmoid({z_i:+.2f}) = {a_i:.2f}")

print("\n💎 OR(1, 0) = 1 ✓   AND(1, 0) = 0 ✓")


### 🎯 TRY IT #2 — Understanding matrix multiplication

> Notice that `X @ W1` is **matrix multiplication**. This is why **GPU and TPU** hardware matters for neural networks: they are optimised for matrix multiplication.

Question: if the input layer has 100 neurons and the hidden layer has 50, how many multiplications happen in one forward pass?
**Answer:** 100 × 50 = 5000. With a batch of 64 examples, that is 64 × 5000 = 320,000 multiplications in one forward pass.

> 💎 This is exactly why a **TPU**, a specialised matrix multiplication unit, gives such a large speed-up for neural networks. This links back to **A4.1.2 hardware**.


## Part 3 · MNIST — the "Hello World" of neural networks

> **MNIST**: 60,000 handwritten digits for training, each 28×28 grayscale pixels, plus 10,000 test images.
> Task: classify digits 0-9.


In [ ]:
# Load MNIST, built into Keras
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

print(f"📊 Train: {X_train.shape}, labels {y_train.shape}")
print(f"📊 Test:  {X_test.shape}, labels {y_test.shape}")
print("\n   Each image: 28×28 pixels")
print("   Values: 0-255 (grayscale)")
print("   Classes: 0-9 (10 digits)")

# Visualisation: first 10 digits
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i], cmap='gray')
    ax.set_title(f'Label: {y_train[i]}', fontsize=12)
    ax.axis('off')
plt.suptitle('MNIST — first 10 examples from the training set', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


In [ ]:
# Class distribution
fig, ax = plt.subplots(figsize=(10, 4))
unique, counts = np.unique(y_train, return_counts=True)
ax.bar(unique, counts, color='steelblue', edgecolor='black')
ax.set_xticks(unique)
ax.set_xlabel('Digit (class)'); ax.set_ylabel('Count in training set')
ax.set_title('MNIST class distribution (well balanced)',
             fontsize=12, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()
print("\n💡 All 10 classes are represented at similar levels (~5500-6700). Good dataset balance.")


### Preprocessing — normalisation

> Divide pixels by 255 so values become [0, 1]. This is **critical** for neural networks.


In [ ]:
# Pixel normalisation: [0, 255] → [0, 1]
X_train_norm = X_train.astype('float32') / 255.0
X_test_norm  = X_test.astype('float32')  / 255.0

print(f"Before normalisation: min={X_train.min()}, max={X_train.max()}")
print(f"After:                min={X_train_norm.min()}, max={X_train_norm.max():.4f}")


## Part 4 · ANN on MNIST with Keras

> Build a simple ANN: input(784) → hidden(128, ReLU) → output(10, softmax)


In [ ]:
# ANN architecture
ann_model = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),         # 28×28 → 784
    layers.Dense(128, activation='relu'),          # hidden layer
    layers.Dense(10, activation='softmax'),        # output: 10 classes
])

ann_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

ann_model.summary()
print("\n💎 Parameter count:")
print("   Flatten: 0, just reshape")
print("   Dense(128): 784 × 128 + 128 = 100,480")
print("   Dense(10):  128 × 10 + 10 = 1,290")
print("   TOTAL: 101,770 trainable parameters")


In [ ]:
# Train ANN: 5 epochs is enough for the workshop
print("🎓 Training ANN... (5 epochs)\n")
history_ann = ann_model.fit(
    X_train_norm, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    verbose=2
)

# Evaluate on test set
test_loss_ann, test_acc_ann = ann_model.evaluate(X_test_norm, y_test, verbose=0)
print(f"\n✅ Test accuracy: {test_acc_ann:.4f}  ({test_acc_ann*100:.2f}%)")


In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history_ann.history['loss'], 'o-', label='Train loss', color='steelblue')
axes[0].plot(history_ann.history['val_loss'], 's-', label='Val loss', color='orange')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('ANN — Loss', fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history_ann.history['accuracy'], 'o-', label='Train acc', color='steelblue')
axes[1].plot(history_ann.history['val_accuracy'], 's-', label='Val acc', color='orange')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('ANN — Accuracy', fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()


## Part 5 · CNN on MNIST — the real magic

> Now we add **convolutional + pooling** layers. We expect **higher accuracy**.


In [ ]:
# CNN needs 4-D input shape: (batch, height, width, channels)
X_train_cnn = X_train_norm.reshape(-1, 28, 28, 1)
X_test_cnn  = X_test_norm.reshape(-1, 28, 28, 1)

print(f"CNN input shape: {X_train_cnn.shape}")


In [ ]:
# CNN architecture: 2 convolutional blocks + dense output
cnn_model = keras.Sequential([
    # Conv Block 1
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),

    # Conv Block 2
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Fully connected
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax'),
])

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()


In [ ]:
# Train CNN: 5 epochs may take 2-5 minutes on CPU
print("🎓 Training CNN... (5 epochs, may take a few minutes)\n")
history_cnn = cnn_model.fit(
    X_train_cnn, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    verbose=2
)

test_loss_cnn, test_acc_cnn = cnn_model.evaluate(X_test_cnn, y_test, verbose=0)
print(f"\n✅ CNN Test accuracy: {test_acc_cnn:.4f}  ({test_acc_cnn*100:.2f}%)")
print(f"   ANN was: {test_acc_ann:.4f}")


In [ ]:
# Compare ANN vs CNN
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history_ann.history['val_accuracy'], 'o-', label='ANN', color='steelblue', linewidth=2)
axes[0].plot(history_cnn.history['val_accuracy'], 's-', label='CNN', color='green', linewidth=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Validation Accuracy')
axes[0].set_title('ANN vs CNN — Accuracy', fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Bar chart of final results
models_names = ['ANN', 'CNN']
test_accs = [test_acc_ann, test_acc_cnn]
bars = axes[1].bar(models_names, test_accs, color=['steelblue', 'green'], edgecolor='black')
axes[1].set_ylabel('Test Accuracy'); axes[1].set_ylim(0.9, 1.0)
axes[1].set_title('Test Accuracy — final result', fontweight='bold')
for bar, acc in zip(bars, test_accs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                  f'{acc:.4f}', ha='center', fontsize=12, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout(); plt.show()

print(f"\n💎 CNN beats ANN by {(test_acc_cnn - test_acc_ann)*100:.2f} percentage points")
print("   This is because CNN preserves the SPATIAL structure of the image,")
print("   while ANN first flattens everything into a 784-dimensional vector, losing spatial information.")


## Part 6 · CNN Error Analysis

> Where does CNN make mistakes? We will show 9 incorrectly classified cases.


In [ ]:
# CNN predictions
predictions = cnn_model.predict(X_test_cnn, verbose=0)
predicted_classes = predictions.argmax(axis=1)

# Find errors
wrong_idx = np.where(predicted_classes != y_test)[0]
print(f"❌ CNN made {len(wrong_idx)} errors out of {len(y_test)} ({len(wrong_idx)/len(y_test)*100:.2f}%)")

# Show first 9 errors
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    if i < len(wrong_idx):
        idx = wrong_idx[i]
        ax.imshow(X_test[idx], cmap='gray')
        confidence = predictions[idx][predicted_classes[idx]]
        ax.set_title(f'True: {y_test[idx]} · Predicted: {predicted_classes[idx]}\nConfidence: {confidence:.2%}',
                     fontsize=10, color='darkred')
        ax.axis('off')
plt.suptitle('Where does CNN make mistakes? Usually messy or unusual digits',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()


In [ ]:
# Confusion matrix for CNN
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_test, predicted_classes)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10), cbar=False, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — CNN (10 MNIST classes)',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


### 🎯 TRY IT #3 — Which digits are confused most often?

Look at the confusion matrix. Which pairs are most often confused?

> 💡 Usually: 4 ↔ 9, 3 ↔ 8, 5 ↔ 6, because they have **similar visual patterns**.


## Part 7 · IB-style conclusion

### 📝 What we did today (for an IA report)

```
1. NUMPY IMPLEMENTATION:
   - Implemented the forward pass of one neuron using np.dot()
   - Built a simple ANN for logic gates by hand
   - Understood that forward propagation = matrix multiplication + activation

2. KERAS / TENSORFLOW:
   - Built an ANN: Flatten → Dense(128, ReLU) → Dense(10, Softmax)
   - Built a CNN: Conv→Pool→Conv→Pool→Flatten→Dense→Output
   - Trained both on MNIST (5 epochs, batch_size=64, Adam optimiser)

3. EVALUATION:
   - ANN test accuracy: about 98%
   - CNN test accuracy: about 99%, **higher** because it preserves spatial structure
   - Analysed errors using a confusion matrix
```

### 💎 Final workshop secrets

1. **Forward pass = matrix multiplication.** This is why GPU/TPU matter.
2. **Normalisation is required** for neural networks. Use `/ 255.0` for images.
3. **`Sparse categorical crossentropy`** is the loss for multi-class classification with integer labels.
4. **CNN is usually better than ANN on images** because it preserves spatial structure.
5. **Confusion matrix** is the gold standard for analysing multi-class models.
6. **`Conv → ReLU → MaxPool` block** is a standard CNN pattern.

---

### 🏠 Homework (see `Week4_HW2_Practice.ipynb`)

> Architecture experiment: change the number of hidden layers and compare ReLU vs Sigmoid. Record how this affects learning speed and final loss.

---

> 🎓 **Final advice:**
> This workshop is strong preparation for an IA on neural networks. Keep the code; you may need it in the final year.
> Many level 6 and 7 ML IAs are exactly this: architecture experiments + result analysis.
